In [1]:
import pandas as pd
import altair as alt

In [2]:
df = pd.read_csv("Food_Inspections_20260514.csv", engine="python", error_bad_lines=False)
df.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
0,2636632,11 DINING,11 DINING,2658510.0,Restaurant,Risk 1 (High),600 S PAULINA ST,CHICAGO,IL,60612.0,05/12/2026,Canvass,Pass w/ Conditions,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.874093,-87.669256,"(41.87409300990204, -87.66925602002334)"
1,2636649,TAQUERIA HUENTITAN,TAQUERIA HUENTITAN RESTAURANT,2073681.0,Restaurant,Risk 1 (High),4019 W NORTH AVE,CHICAGO,IL,60639.0,05/12/2026,Canvass,Pass,39. CONTAMINATION PREVENTED DURING FOOD PREPAR...,41.909763,-87.727291,"(41.90976347642956, -87.72729104153534)"
2,2636611,TUTORE ITALIAN COOKING,TUTORE ITALIAN COOKING,3082471.0,NaN,Risk 1 (High),1629 W GRAND AVE,CHICAGO,IL,60622.0,05/12/2026,License,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.890856,-87.668303,"(41.89085551482435, -87.66830289784055)"
3,2636619,BELL HEIRS BBQ,BELL HEIRS BBQ,3082833.0,Restaurant,Risk 1 (High),131 N CLINTON ST,CHICAGO,IL,60661.0,05/12/2026,License,Pass,51. PLUMBING INSTALLED; PROPER BACKFLOW DEVICE...,41.884188,-87.641120,"(41.884187507127805, -87.64111966683218)"
4,2636671,Taco Bell,Taco Bell,2983552.0,Restaurant,Risk 1 (High),4523 W NORTH AVE,CHICAGO,IL,60639.0,05/12/2026,Canvass,Pass,"38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - ...",41.909596,-87.739701,"(41.90959614296241, -87.73970064401364)"


In [3]:
df.columns

Index(['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type',
       'Risk', 'Address', 'City', 'State', 'Zip', 'Inspection Date',
       'Inspection Type', 'Results', 'Violations', 'Latitude', 'Longitude',
       'Location'],
      dtype='object')

In [4]:
df = df[["Inspection ID", "Facility Type", "Risk", "Inspection Date", "Results"]]

In [5]:
df = df.rename(columns={"Inspection ID": "inspection_id", "Facility Type": "facility_type", "Risk": "risk", "Inspection Date": "inspection_date", 
                            "Results": "results"})

In [6]:
df_clean = df.copy()

df_clean["inspection_date"] = pd.to_datetime(df_clean["inspection_date"], errors="coerce")
df_clean["year"] = df_clean["inspection_date"].dt.year

df_clean = df_clean.dropna(subset=["facility_type", "results", "year"])

df_clean["facility_type"] = df_clean["facility_type"].str.title().str.strip()
df_clean["results"] = df_clean["results"].str.title().str.strip()
df_clean["risk"] = df_clean["risk"].astype(str).str.title().str.strip()

df_clean = df_clean[
    (df_clean["facility_type"] != "") &
    (df_clean["facility_type"].str.lower() != "nan") &
    (df_clean["facility_type"].str.lower() != "none")
]

df_clean.head()

,inspection_id,facility_type,risk,inspection_date,results,year
0,2636632,Restaurant,Risk 1 (High),2026-05-12,Pass W/ Conditions,2026
1,2636649,Restaurant,Risk 1 (High),2026-05-12,Pass,2026
3,2636619,Restaurant,Risk 1 (High),2026-05-12,Pass,2026
4,2636671,Restaurant,Risk 1 (High),2026-05-12,Pass,2026
5,2636629,Restaurant,Risk 1 (High),2026-05-12,Fail,2026


In [7]:
top_facilities = df_clean["facility_type"].value_counts().head(15).index

df_clean = df_clean[df_clean["facility_type"].isin(top_facilities)]

df_clean["failed"] = 0
df_clean.loc[df_clean["results"] == "Fail", "failed"] = 1

failure_summary = df_clean.groupby(["year", "facility_type"], as_index=False).agg({"inspection_id": "count", "failed": "sum"})

failure_summary = failure_summary.rename(columns={
    "inspection_id": "total_inspections",
    "failed": "failed_inspections"
})

failure_summary["failure_rate"] = (failure_summary["failed_inspections"] / failure_summary["total_inspections"] * 100)

failure_summary.head()

,year,facility_type,total_inspections,failed_inspections,failure_rate
0,2010,Bakery,225,62,27.555556
1,2010,Catering,97,28,28.865979
2,2010,Children'S Services Facility,206,35,16.990291
3,2010,Daycare (2 - 6 Years),427,129,30.210773
4,2010,Daycare Above And Under 2 Years,254,56,22.047244


In [8]:
failure_summary_plot = failure_summary[failure_summary["failed_inspections"] > 0].copy()

years = sorted(failure_summary_plot["year"].dropna().unique())

year_dropdown = alt.binding_select(
    options=years,
    name="Select year: "
)

year_selection = alt.selection_point(
    fields=["year"],
    bind=year_dropdown,
    value={"year": max(years)}
)

visual1 = (
    alt.Chart(failure_summary_plot)
    .mark_bar()
    .encode(
        x=alt.X(
            "failure_rate:Q",
            title="Percent of Inspections That Failed"
        ),
        y=alt.Y(
            "facility_type:N",
            title="Facility Type",
            sort="-x"
        ),
        color=alt.Color(
            "failure_rate:Q",
            title="Failure Rate (%)",
            scale=alt.Scale(scheme="orangered")
        ),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("facility_type:N", title="Facility Type"),
            alt.Tooltip("total_inspections:Q", title="Total Inspections"),
            alt.Tooltip("failed_inspections:Q", title="Failed Inspections"),
            alt.Tooltip("failure_rate:Q", title="Failure Rate (%)", format=".2f")
        ]
    )
    .add_params(year_selection)
    .transform_filter(year_selection)
    .properties(
        width=750,
        height=500,
        title="Food Inspection Failure Rates by Facility Type in Chicago"
    )
)

visual1

alt.Chart(...)

In [9]:
visual1.save("inspection_failure_rates.html")

**Contextual Visualization 1: Active Retail Food Establishments by Chicago Ward**

In [10]:
alt.data_transformers.disable_max_rows()
retail = pd.read_csv("Retail_Food_Establishments__Chicago_20260514.csv")
retail.head()

,ACCOUNT NUMBER,SITE NUMBER,LEGAL NAME,DOING BUSINESS AS NAME,ADDRESS,CITY,STATE,ZIP CODE,WARD,PRECINCT,...,LICENSE NUMBER,APPLICATION TYPE,PAYMENT DATE,LICENSE TERM START DATE,LICENSE TERM EXPIRATION DATE,DATE ISSUED,LICENSE STATUS,LATITUDE,LONGITUDE,LOCATION
0,335347,1,SD Trading Corp.,01 Food Mart,10501 S WENTWORTH AVE 1ST STO,CHICAGO,IL,60628.0,9.0,26.0,...,1922247,RENEW,03/22/2014,09/16/2013,09/15/2015,03/24/2014,AAI,41.703415,-87.628031,POINT (-87.6280313854684 41.70341510444771)
1,335347,1,SD Trading Corp.,01 Food Mart,10501 S WENTWORTH AVE 1ST STO,CHICAGO,IL,60628.0,9.0,26.0,...,1922247,RENEW,05/02/2012,09/16/2011,09/15/2013,05/02/2012,AAI,41.703415,-87.628031,POINT (-87.6280313854684 41.70341510444771)
2,302,1,"1000 LIQUOR'S, INC.",1000 LIQUORS / BIG CITY TAP,1000-1012 W BELMONT AVE 1ST,CHICAGO,IL,60657.0,44.0,39.0,...,1578,RENEW,12/30/2024,02/16/2025,02/15/2027,12/31/2024,AAI,41.940020,-87.654172,POINT (-87.65417210636889 41.940020360226804)
3,302,1,"1000 LIQUOR'S, INC.",1000 LIQUORS / BIG CITY TAP,1000-1012 W BELMONT AVE 1ST,CHICAGO,IL,60657.0,44.0,39.0,...,1578,RENEW,02/11/2023,02/16/2023,02/15/2025,05/09/2023,AAI,41.940020,-87.654172,POINT (-87.65417210636889 41.940020360226804)
4,302,1,"1000 LIQUOR'S, INC.",1000 LIQUORS / BIG CITY TAP,1000-1012 W BELMONT AVE 1ST,CHICAGO,IL,60657.0,44.0,39.0,...,1578,RENEW,02/11/2021,02/16/2021,02/15/2023,02/25/2021,AAI,41.940020,-87.654172,POINT (-87.65417210636889 41.940020360226804)


In [11]:
retail_clean = retail.copy()

retail_clean.columns = retail_clean.columns.str.strip()
retail_clean["WARD"] = pd.to_numeric(retail_clean["WARD"], errors="coerce")
retail_clean = retail_clean.dropna(subset=["WARD"])
retail_clean["WARD"] = retail_clean["WARD"].astype(int)

ward_summary = (
    retail_clean.groupby("WARD")
    .agg(total_food_establishments=("ACCOUNT NUMBER", "count"))
    .reset_index()
    .sort_values("WARD")
)

ward_summary.head()

,WARD,total_food_establishments
0,1,2956
1,2,2850
2,3,1129
3,4,1665
4,5,1187


In [12]:
hover = alt.selection_point(
    fields=["WARD"],
    on="mouseover",
    clear="mouseout"
)

ward_chart = alt.Chart(ward_summary).mark_bar().encode(
    x=alt.X(
        "WARD:O",
        title="Chicago Ward",
        sort="ascending"
    ),
    y=alt.Y(
        "total_food_establishments:Q",
        title="Number of Active Retail Food Establishments"
    ),
    opacity=alt.condition(hover, alt.value(1), alt.value(0.65)),
    tooltip=[
        alt.Tooltip("WARD:O", title="Ward"),
        alt.Tooltip("total_food_establishments:Q", title="Active Food Establishments")
    ]
).add_params(
    hover
).properties(
    width=750,
    height=400,
    title="Where Are Chicago’s Active Food Businesses Concentrated?"
)

ward_chart

alt.Chart(...)

In [13]:
ward_chart.save("retail_food_establishments_by_ward.html")

**Contextual Visualization 2: Number of Grocery Stores in Zip Codes**

In [14]:
grocery = pd.read_csv("Grocery_Store_Status_-_Historical_20260514.csv")
grocery.head()

,Store Name,Address,Zip,New status,Last updated,Location
0,Jewel - Osco,87 W 87th St,60620,OPEN,06/03/2020 05:00:00 PM,POINT (-87.626243 41.736172)
1,Farm on Ogden,3555 W OGDEN AVE,60623,OPEN,06/10/2020 12:00:00 AM,POINT (-87.71437 41.854608)
2,Jewel - Osco,5343 N Broadway St,60640-2311,OPEN,06/03/2020 05:00:00 PM,POINT (-87.659887 41.978998)
3,International Foods NW,4404 W FULLERTON AVE,60639,OPEN,06/10/2020 12:00:00 AM,POINT (-87.737127 41.924425)
4,Jewel - Osco,2520 N Narragansett Ave,60639-1041,OPEN,06/03/2020 05:00:00 PM,POINT (-87.785559 41.926236)


In [15]:
grocery.columns

Index(['Store Name', 'Address', 'Zip', 'New status', 'Last updated',
       'Location'],
      dtype='object')

In [16]:
grocery_clean = grocery.copy()
grocery_clean = grocery_clean[["Store Name", "Address", "Zip", "New status", "Last updated"]].copy()
grocery_clean["Zip"] = grocery_clean["Zip"].astype(str).str[:5]
grocery_clean = grocery_clean[grocery_clean["New status"] == "OPEN"]

grocery_zip_summary = (
    grocery_clean
    .groupby("Zip")
    .size()
    .reset_index(name="number_of_grocery_stores")
    .sort_values("number_of_grocery_stores", ascending=False)
    .head(15)
)

grocery_zip_summary.head()

,Zip,number_of_grocery_stores
32,60639,14
40,60647,14
4,60608,13
27,60632,10
10,60614,10


In [17]:
grocery_chart = (
    alt.Chart(grocery_zip_summary)
    .mark_bar()
    .encode(
        x=alt.X(
            "number_of_grocery_stores:Q",
            title="Number of Open Grocery Stores",
            axis=alt.Axis(titleFontSize=15, labelFontSize=12)
        ),
        y=alt.Y(
            "Zip:N",
            title="ZIP Code",
            sort="-x",
            axis=alt.Axis(titleFontSize=15, labelFontSize=12)
        ),
        tooltip=[
            alt.Tooltip("Zip:N", title="ZIP Code"),
            alt.Tooltip("number_of_grocery_stores:Q", title="Number of Open Grocery Stores")
        ]
    )
    .properties(
        width=750,
        height=450,
        title={
            "text": "Where Open Grocery Stores Are Concentrated in Chicago",
            "fontSize": 20,
            "subtitleFontSize": 14,
            "anchor": "middle"
        }
    )
    .configure_view(strokeWidth=0)
)

grocery_chart

alt.Chart(...)

In [18]:
grocery_chart.save("grocery_store_status_by_zip.html")

**Contextual Visualization 3**

In [19]:
# Extra credit

result_counts = (
    df_clean
    .groupby("results")
    .agg(total_inspections=("inspection_id", "count"))
    .reset_index()
    .sort_values("total_inspections", ascending=False)
)

result_chart = alt.Chart(result_counts).mark_bar().encode(
    x=alt.X("total_inspections:Q", title="Number of Inspections"),
    y=alt.Y("results:N", sort="-x", title="Inspection Result"),
    tooltip=[
        alt.Tooltip("results:N", title="Inspection Result"),
        alt.Tooltip("total_inspections:Q", title="Total Inspections")
    ]
).properties(
    width=750,
    height=400,
    title="What Results Do Chicago Food Inspections Usually Receive?"
)

result_chart

alt.Chart(...)

In [20]:
result_chart.save("inspection_results_breakdown.html")